# SFINCS — NJ Sandy: results & methodology viewer

A **visualization-only** companion to the build/run pipeline. The model is built
and run by the experiment harness (`run_experiments.py` + the `nj_sfincs`
package); this notebook just **opens a finished run read-only and plots it**, so
you can show any methodological choice on demand — the elevation model, the grid,
the mask, each forcing, the flood map, and the validation against the Sandy Hook
gauge / USGS High Water Marks / the FEMA MOTF extent.

> Pick which run to view by setting **`EXP`** in the setup cell.

---

## ⚠️ Read this first (2026-07-14) — the domain was broken, and is now rebuilt

Everything in this project before 14 July ran on a domain with **two holes in its
plumbing**, and both are now fixed at the root:

1. **The model was draining the estuary.** `region.geojson` cut the Navesink River in
   half mid-channel, so hydromt placed a *free-outflow boundary* — a drain — on a
   five-metre-deep tidal cross-section. **92.6% of all the water entering the estuary
   flowed straight out of the domain.**
2. **Shark River Inlet was dammed shut.** The top-priority 2010 topobathy lidar failed
   to penetrate the turbid inlet and returned the **water surface** (+0.4 to +2.2 m)
   instead of the bed, shadowing CUDEM's correct −3 m. The whole Shark estuary therefore
   **never flooded in any run — peak level exactly +0.00 m** — while the ocean 1.8 km
   away reached +2.9 m. (It is *not* a bridge: the dam's edge is the edge of the lidar tile.)

Both defects are **infrastructure — a region polygon and an elevation tier — not physics.**
That is exactly why two months of eliminating every *physical* lever (wind, friction, mesh
resolution, dredging, wave convergence) came back null.

**Consequence for this notebook:** every pre-rebuild experiment hard-links the same broken
`sfincs.nc`, so the old engine/knob comparisons are **explained, not informative** — do not
cite them as physics. The sealed runs are named `sealed_*`. The verification section at the
bottom is the one that matters.

*Full write-up: `reports/shrewsbury_investigation.md` (Workstreams J, K, L).*

## Setup

In [ ]:
# Import the viz stack up top (this also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Make the nj_sfincs package importable from notebooks/.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate
from nj_sfincs.config import EXPERIMENTS

# ── Choose the run to view ───────────────────────────────────────────────────
# SEALED domain (2026-07-14 rebuild — the leak plugged, Shark River Inlet carved open):
#     sealed_faber_waves     sealed_faber_nowaves
#     sealed_galibier_waves  sealed_galibier_nowaves
#
# PRE-REBUILD runs (snapwave_tuned*, leakfix_*, faber_*) are on the BROKEN domain —
# they are kept only as the "before" for the verification section. Do not cite them.
EXP = "sealed_faber_waves"

exp_dir = ROOT / "experiments" / EXP
if not exp_dir.exists():
    exp_dir = ROOT / "model"  # fall back to the reference build
    print(f"experiments/{EXP} not found — showing the reference model/ build")
print("viewing:", exp_dir)

In [ ]:
# Open the run twice: `sf` (read mode, for the build + forcing inputs) and
# `mod` (for the solver output), and downscale the flood map once.
DATA_LIBS = [str(ROOT / "data" / "data_catalog.yml")]
sf = SfincsModel(str(exp_dir), data_libs=DATA_LIBS, mode="r")
sf.read()   # repopulates grid + every forcing component from disk

mod, da_hmax, da_dep = validate.load_floodmap(exp_dir)
print("output vars:", list(mod.output.data.keys()))

---
## Methodology — the static build

These document the modeling choices: the quadtree grid, the merged topobathy,
and the active/boundary mask.

### The quadtree grid

In [ ]:
plots.plot_grid(sf);

### Topobathy (interactive) — pan/zoom the dunes, inlets, dredged channels

In [ ]:
plots.plot_topobathy(sf)

### The mask — active interior, water-level boundary, outflow

In [ ]:
plots.plot_mask(sf);

---
## Methodology — the forcing

The compound drivers, read back from the written model: the observed surge
boundary, ERA5 wind/pressure, AORC rainfall, and USGS river discharge.

### Surge boundary (NOAA CO-OPS)

In [ ]:
plots.plot_surge(sf);

### Wind + pressure (ERA5)

In [ ]:
plots.plot_wind_pressure(sf);

### Rainfall (NOAA AORC)

In [ ]:
plots.plot_rain(sf);

### River discharge (USGS)

In [ ]:
plots.plot_discharge(sf);

---
## Results

The SnapWave field (if this run has waves), the downscaled flood map, and the
three validations.

### SnapWave Hm0 at peak — look for a lee behind Sandy Hook

In [ ]:
res = plots.plot_wave_field(mod)
if res is None:
    print("no wave output for this run (waves off, or no hm0)")

### Maximum flood depth

In [ ]:
plots.plot_floodmap(mod, da_hmax);

### Validation 1 — Sandy Hook gauge (temporal)

In [ ]:
m = validate.gauge_peak_error(mod)
print(f"observed peak: {m['gauge_obs_peak_m']:.2f} m | modeled peak: "
      f"{m['gauge_mod_peak_m']:.2f} m | error: {m['gauge_peak_err_m']:+.2f} m")

### Validation 2 — USGS High Water Marks (spatial)

In [ ]:
plots.plot_hwm_scatter(da_hmax, da_dep);

In [ ]:
plots.plot_hwm_residual_map(mod, da_hmax, da_dep);

### Validation 3 — FEMA MOTF extent (CSI / POD / FAR)

In [ ]:
plots.plot_motf(da_hmax, da_dep);

---
## Compare the wave experiments

Reads `experiments/metrics.csv` (written by `run_experiments.py`). Higher CSI /
POD and lower FAR / HWM-RMSE are better; `shb_hm0` is the SnapWave Hm0 in the
Sandy Hook Bay lee — the "did waves reach the bay?" number.

> ⚠️ **This table is from the BROKEN domain** (pre-2026-07-14) unless you have re-run
> `run_experiments.py` since the rebuild. Every row in it was scored on a model that was
> draining 92.6% of the estuary's inflow and had Shark River sealed shut, so the *rankings*
> between wave presets are not meaningful — they were all asking why a leaking bucket would
> not fill. Use the **sealed-domain verification** section below instead.
>
> Note also that `hwm_bias_m` here is the **legacy wet-only** metric, which silently drops any
> mark the model fails to flood and therefore *rewards under-flooding*. The repaired metric
> (`hwm_bias_scored_m`) scores dry marks at ground level. See `nj_sfincs/validate.py`.

In [ ]:
metrics_csv = ROOT / "experiments" / "metrics.csv"
if metrics_csv.exists():
    metrics = pd.read_csv(metrics_csv, index_col=0)
    display(metrics.round(3))
else:
    metrics = None
    print("No experiments/metrics.csv yet — run:  python run_experiments.py")

In [ ]:
if metrics is not None:
    plots.plot_experiment_comparison(metrics, ROOT / "experiments" / "floodmaps");

---

# Sealed-domain verification (2026-07-14)

> The previous section here — *"Engine comparison — Faber vs Galibier (Workstream I)"* — has
> been removed. It ran on the leaking, dammed domain, and its conclusion (that a ~1.2 m spread
> between engines was "physics, not numerics") was measured on a model that was draining 92.6%
> of the estuary's inflow. Those runs have been deleted; the engine question is being re-asked
> on the sealed domain (`sealed_*`). Its final paragraph also blamed the residual on **barrier
> erosion / missing morphodynamics** — that hypothesis was **withdrawn** (the Sea Bright seawall
> held; there was no breach there).

## Verification 1 — the gauges, and specifically **the tide**

**This is the headline test, and it needs no storm peak and no high-water marks.**

Both interior USGS gauges died on **2012-10-29 03:54**, roughly 20 h before Sandy's peak, so
neither has a storm crest — the Shrewsbury "2.935 m" is a *post-event surveyed mark*, drawn as a
single star rather than a curve. That sounds like weak validation, and for the surge it is.

But it is exactly the right instrument for the two bugs we just fixed, because what the record
*does* contain is a clean **pre-storm tide**, and **both defects destroy the tide**:

* the Navesink leak drained the estuary from a flat start, so the model fell monotonically
  instead of oscillating;
* the Shark River Inlet dam cut that basin off from the ocean entirely, so it **never oscillated
  at all** — fraction of time rising **0.00**, against **0.47** observed.

So read the **left** half of these panels. A tide floods *and* ebbs; a drained or dammed basin
only ebbs.

| | observed | every pre-rebuild run | **sealed + carved** |
|---|---|---|---|
| Shark tidal range | **1.52 m** | *none — never oscillated* | **1.331 m** |
| Shark, fraction rising | **0.47** | **0.00** | **0.542** |
| Shrewsbury tidal range | 1.23 m | 0.716 | **0.996** |

In [ ]:
# Observed vs modelled water level at the two interior gauges.
# Read the PRE-STORM TIDE (left of the dotted "gauge dies" line).
#
# The BROKEN and LEAK-FIX-ONLY traces flatline at Shark — they sink and sit there,
# because the inlet was dammed shut. Only the SEALED run oscillates.
VERIFY_RUNS = {
    "BROKEN premier (leaking + dammed)": "snapwave_tuned_25m",
    "leak fix only (mask edit)":         "leakfix_extend_nowaves_25m",
    "SEALED + CARVED domain":            "sealed_faber_nowaves",
}
# swap in sealed_faber_waves once the wave arm lands, to see setup on top of the tide
plots.plot_gauge_verification(VERIFY_RUNS);

## Verification 2 — flood extent vs FEMA MOTF

**Read the CSI, not the POD.** FEMA MOTF is a HWM/sensor-interpolated *bathtub* surface that
shares provenance with our own high-water marks (a flat 3.4 m fill reproduces it at IoU 0.906),
so it is an extent **consistency** check, not an independent observation. And its **POD
structurally rewards over-flooding** — flood everything and you score a perfect POD. It is the
mirror image of the HWM-bias flaw, which rewards *under*-flooding. Lead with the gauge +
HWM + tidal-range trio; use this to see *where* the extent differs.

That caveat matters here, because it is what makes the result credible:

| | CSI | POD | FAR | hit | miss |
|---|---|---|---|---|---|
| broken premier, **with** waves | 0.51 | 0.56 | 0.17 | 19.7 km² | 15.3 km² |
| sealed + carved, **no** waves | **0.64** | **0.72** | **0.14** | **25.1 km²** | **9.9 km²** |

The blue *miss* blobs in the back-bays have turned green. And the **false-alarm rate went down**
(0.17 → 0.14), so this is not the model over-flooding to game the metric that rewards exactly
that. A run with the waves *switched off* now beats the old premier with them on.

In [ ]:
# Modeled flood vs FEMA MOTF — hit / miss / false-alarm, side by side.
MOTF_RUNS = {
    "BROKEN premier (waves)":    "snapwave_tuned_25m",
    "SEALED + CARVED (waves)":   "sealed_faber_waves",
}
plots.plot_motf_panels(MOTF_RUNS);

## Verification 3 — the premier, re-asked on a domain that isn't broken

Faber vs Galibier, waves on and off, on the sealed + carved domain.

**Galibier carries `snapwave_gammax = 2.0` restored.** Galibier's default is 999, which
*removes* the per-sweep wave-height clamp and produces a 252 m runaway wave at the Sea Bright
bay mouth. That finding is about the **source code**, not the domain, so it survives the
rebuild — and without the clamp we would be comparing Faber's physics against Galibier's
instability rather than against Galibier's physics.

**The test this has to pass — and it is the one that could still sink it:** a domain fix is
**local**. South-coast HWM bias was **−0.0553** on the broken domain and stayed **−0.0553**
through the leak fix. **If the open coast moves now, we changed something we did not mean to,
and the rest of the result is not trustworthy.**

### What is still open, after this

* **The +1.03 m Atlantic-oceanfront bias**, which the fixes left standing (it was +0.73 m
  before). This is now the **largest remaining error in the model**, and unlike everything
  above it is a genuine *physics* question, not a plumbing one.
* **The open-boundary depth** (James's suggestion): sweep `mask_zmin` −10 / −15 / −20. Needs no
  rebuild — the mesh reaches −69 m and every face already has subgrid tables.

*Wind (D) is retired, not re-run: it measured +0.002 m, which is not "the leak ate it" but no
forcing response at all — a few km of fetch over a 3.8e7 m³ prism gives wind no mechanism to
act through whether the bucket holds or not.*

In [ ]:
# The sealed premier 2x2 — the full table (tide, gauge, per-basin HWM bias).
# Produced by scripts/analyze_sealed.py; re-run that script if the CSV is stale.
sealed_csv = ROOT / "reports" / "sealed_premier.csv"
if sealed_csv.exists():
    sealed = pd.read_csv(sealed_csv)
    cols = [c for c in ["desc", "shark_frac_rising", "shark_tide", "shrews_tide",
                        "gauge", "gauge_err", "shrewsbury_navesink", "shark_river",
                        "south_coast", "atlantic_oceanfront", "rmse"] if c in sealed]
    display(sealed[cols].round(3))
    print("\nTHE LOCALITY TEST: south_coast must stay at -0.055.")
    print("If it moved, the rebuild changed something we did not intend.")
else:
    print("run:  NJ_ROOT=$PWD PYTHONPATH=$PWD python scripts/analyze_sealed.py")